# Lab 3 : Visualiser les données 311

> **Avant de commencer - choisir votre support**
> Téléchargez un seul format avant de travailler :
>
> - **Notebook `.ipynb`** pour Google Colab, JupyterLab, Jupyter Notebook ou VS Code : ouvrez le dossier GitHub [`labsheets/`](https://github.com/iaousse/dldv-course/tree/main/labsheets) et téléchargez le fichier source du  [Lab 3](https://github.com/iaousse/dldv-course/tree/main/labsheets/lab3_visualisation_donnees.ipynb).
> - **Fichier `.qmd`** pour travailler avec Quarto : ouvrez le dossier GitHub [`tp/`](https://github.com/iaousse/dldv-course/tree/main/tp) et téléchargez le fichier source du  [Lab 3](https://github.com/iaousse/dldv-course/tree/main/tp/lab3_visualisation_donnees.qmd).

> **Objectifs du Lab**
> À la fin de ce Lab, l'étudiant sera capable de :
>
> - **Choisir** un type de graphique adapté à une question de distribution, de comparaison, de relation, d'évolution ou de composition ;
> - **Construire** les graphiques essentiels avec `matplotlib` et `seaborn` à partir d'une table déjà nettoyée ;
> - **Repérer et corriger** un graphique trompeur (axe tronqué, camembert surchargé, échelle mal choisie) ;
> - **Produire** un premier graphique interactif avec `Plotly` ;
> - **Concevoir** une figure de communication complète : titre-message, annotation, source, palette accessible ;
> - **Organiser** un petit portfolio de graphiques reproductible, prêt à être présenté à un public non technique.

> **Idée centrale**
> Le Lab 2 a produit une table propre. Ce Lab ne rajoute aucune nouvelle donnée : il transforme cette table en une série de graphiques qui **répondent chacun à une question précise**, en respectant les principes vus au chapitre sur la visualisation des données. Un graphique qui ne répond à aucune question n'a pas sa place dans ce Lab.

## 1. Le fichier de départ

Utilisez `data/processed/nyc311_janvier_clean.csv`, produit au Lab 2. Si vous ne l'avez pas, reprenez `data/raw/nyc311_janvier.csv` et appliquez au minimum la conversion des dates et le calcul de `processing_hours` (voir Lab 2, sections 4 et 5).

Rappel des colonnes disponibles :

| Colonne | Rôle |
|:---|:---|
| `unique_key` | Identifiant de la demande |
| `created_date`, `closed_date` | Dates de création et de clôture |
| `agency` | Agence responsable |
| `complaint_type` | Type de problème déclaré |
| `borough_normalized` | Arrondissement, libellé normalisé |
| `open_data_channel_type` | Canal de transmission |
| `status` | Statut de la demande |
| `processing_hours` | Délai de traitement, en heures |
| `created_weekday`, `created_hour` | Jour de la semaine et heure de création |

La question de communication de ce Lab est directement héritée du Lab 1 :

> Quels types de demandes sont les plus fréquents, et quels facteurs semblent associés à un délai de traitement plus long ?

Le Lab 2 a permis d'explorer cette question avec des chiffres. Ce Lab la traduit en **graphiques présentables**.

## 2. Installer et préparer

```bash
uv add matplotlib seaborn plotly
```


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROCESSED_PATH = Path("../data/processed/nyc311_janvier_clean.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(PROCESSED_PATH, parse_dates=["created_date", "closed_date"])
df.shape



Les dates du fichier sont exprimées en heure locale de New York (voir le Lab 2). Pour les graphiques de délai, ne gardez que les demandes clôturées avec un délai valide : une demande encore ouverte n'a pas encore de durée complète observable.


In [ ]:
df_delais = df.loc[
    df["processing_hours"].notna()
    & (df["processing_hours"] >= 0)
].copy()

print(f"Demandes dans le fichier : {df['unique_key'].nunique():,}")
print(f"Demandes avec délai valide : {df_delais['unique_key'].nunique():,}")



Adoptez dès maintenant une palette accessible pour l'ensemble du Lab, conformément aux bonnes pratiques du chapitre 3 :


In [ ]:
sns.set_theme(style="whitegrid")
sns.set_palette("colorblind")



> **Un dossier, un graphique, un nom explicite**
> Enregistrez chaque figure dans `outputs/` avec un nom qui décrit son contenu, par exemple `lab3_histogramme_delai.png`, plutôt que `figure1.png`. Vous devrez retrouver rapidement quel fichier correspond à quelle question.

## 3. Distribution : à quoi ressemble le délai de traitement ?

Reprenez la variable `processing_hours` déjà calculée au Lab 2 et construisez un histogramme **et** une boîte à moustaches, côte à côte, comme vu au chapitre 3 (§4.1 et §4.6) :


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df_delais, x="processing_hours", bins=50, ax=axes[0])
axes[0].set_xlim(0, df_delais["processing_hours"].quantile(0.99))
axes[0].set_title("Distribution du délai de traitement (99e percentile)")
axes[0].set_xlabel("Délai (heures)")

sns.boxplot(data=df_delais, x="processing_hours", ax=axes[1])
axes[1].set_title("Délai de traitement — valeurs extrêmes visibles")
axes[1].set_xlabel("Délai (heures)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_distribution_delai.png", dpi=150)



> **Question de vérification**
> Pourquoi l'histogramme est-il limité au 99ᵉ percentile alors que la boîte à moustaches montre toute la variable ? Rédigez une phrase de légende qui explique cette différence à un lecteur non technique.

## 4. Comparaison : quels types de demandes dominent ?

Construisez un diagramme en barres **horizontales et triées**, conformément à la bonne pratique du §4.2 :


In [ ]:
top_types = df["complaint_type"].value_counts().head(12).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=top_types.values, y=top_types.index, ax=ax, orient="h")
ax.set_title("Les 12 types de demandes les plus fréquents — janvier")
ax.set_xlabel("Nombre de demandes")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_top_types_barres.png", dpi=150)



### 4.1. Contre-exemple volontaire : le camembert

Construisez maintenant un camembert des huit types de demandes les plus fréquents, à comparer au diagramme en barres :


In [ ]:
parts = df["complaint_type"].value_counts().head(8)

plt.figure(figsize=(6, 6))
plt.pie(parts.values, labels=parts.index, autopct="%1.0f%%")
plt.title("Répartition des 8 types de demandes les plus fréquents")
plt.savefig(OUTPUT_DIR / "lab3_camembert_a_critiquer.png", dpi=150)



> **À faire**
> Comparez ce camembert au diagramme en barres de la section 4. Lequel des deux permet de répondre le plus vite à la question « quel est le troisième type de demande le plus fréquent, et de combien dépasse-t-il le quatrième ? ». Notez votre réponse pour la partie C du travail demandé.

## 5. Relation : le délai dépend-il du moment de la journée ?

Construisez un nuage de points reliant l'heure de création et le délai de traitement, en limitant l'axe des délais pour rester lisible :


In [ ]:
echantillon = df_delais.sample(n=min(5000, len(df_delais)), random_state=42)

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=echantillon,
    x="created_hour",
    y="processing_hours",
    alpha=0.3,
    ax=ax,
)
ax.set_ylim(0, df_delais["processing_hours"].quantile(0.95))
ax.set_title("Délai de traitement selon l'heure de création de la demande")
ax.set_xlabel("Heure de création")
ax.set_ylabel("Délai (heures)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_nuage_heure_delai.png", dpi=150)



> **Pourquoi échantillonner ?**
> Un nuage de points de plusieurs centaines de milliers de points devient illisible (une bouillie de couleur) et lent à afficher. L'échantillonnage (`sample`) et la transparence (`alpha`) sont deux réponses complémentaires à ce problème — retenez les deux.

Complétez par une matrice de corrélation entre les variables numériques disponibles, sous forme de heatmap (§4.7) :


In [ ]:
numeriques = df_delais[["processing_hours", "created_hour"]].corr()

plt.figure(figsize=(4, 3))
sns.heatmap(numeriques, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Corrélation entre l'heure de création et le délai")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_heatmap_correlation.png", dpi=150)



## 6. Évolution : comment le volume de demandes varie-t-il dans le temps ?


In [ ]:
volume_quotidien = (
    df.set_index("created_date")
    .resample("D")
    .size()
    .rename("nombre_demandes")
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(volume_quotidien.index, volume_quotidien.values, marker="o", markersize=3)
ax.set_title("Nombre de demandes créées par jour — janvier")
ax.set_xlabel("Date")
ax.set_ylabel("Nombre de demandes")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_evolution_quotidienne.png", dpi=150)



> **Rappel du chapitre 3**
> Une courbe suppose un ordre naturel entre les points : ici, le temps. Ne l'utilisez jamais pour relier des catégories comme les types de demandes ou les arrondissements.

## 7. Composition et multivarié : délai par arrondissement et par canal

Une boîte à moustaches par arrondissement, pour comparer des **distributions** plutôt que de simples moyennes :


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=df_delais,
    x="borough_normalized",
    y="processing_hours",
    ax=ax,
)
ax.set_ylim(0, df_delais["processing_hours"].quantile(0.95))
ax.set_title("Délai de traitement par arrondissement")
ax.set_xlabel("")
ax.set_ylabel("Délai (heures)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_boxplot_arrondissement.png", dpi=150)



Puis une heatmap croisant arrondissement et canal de transmission, à partir d'un tableau croisé (vu au chapitre 2) :


In [ ]:
tableau = pd.crosstab(df["borough_normalized"], df["open_data_channel_type"], normalize="index")

plt.figure(figsize=(7, 5))
sns.heatmap(tableau, annot=True, fmt=".0%", cmap="Blues")
plt.title("Répartition des canaux de transmission par arrondissement")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_heatmap_canal_arrondissement.png", dpi=150)



## 8. Un premier graphique interactif

Reproduisez le nuage de points de la section 5 avec `Plotly`, pour découvrir l'infobulle interactive présentée au chapitre 3 (§5.3) :


In [ ]:
import plotly.express as px

fig = px.scatter(
    echantillon,
    x="created_hour",
    y="processing_hours",
    color="borough_normalized",
    hover_data=["complaint_type", "status"],
    title="Délai de traitement selon l'heure de création (interactif)",
    labels={"created_hour": "Heure de création", "processing_hours": "Délai (heures)"},
)
fig.write_html(str(OUTPUT_DIR / "lab3_scatter_interactif.html"))



> **À retenir**
> Ouvrez le fichier `.html` obtenu dans un navigateur : vous pouvez zoomer, filtrer par arrondissement en cliquant sur la légende, et lire le détail d'un point au survol. Aucune de ces actions n'est possible avec la version `matplotlib`.

## 9. Construire une figure de communication complète

Choisissez **un seul** graphique parmi ceux produits aux sections 3 à 7 — celui qui, selon vous, porte le message le plus utile — et transformez-le en véritable figure de communication, en appliquant l'anatomie d'un bon graphique (chapitre 3, §6.1) :

1. Un **titre-message** (pas un titre-thème) : reformulez le titre pour qu'il énonce une conclusion.
2. Une **annotation** directe sur le point ou la barre la plus importante (`ax.annotate(...)`).
3. Une **mention de la source et de la période** en petit texte sous le graphique.
4. Une **palette accessible** (déjà appliquée via `sns.set_palette("colorblind")`).


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_types.values, y=top_types.index, ax=ax)
ax.set_title("Les demandes de bruit dominent largement le service 311 en janvier")
ax.set_xlabel("Nombre de demandes")
ax.set_ylabel("")
ax.annotate(
    "Type le plus fréquent",
    xy=(top_types.values[-1], len(top_types) - 1),
    xytext=(top_types.values[-1] * 0.6, len(top_types) - 3),
    arrowprops=dict(arrowstyle="->"),
)
fig.text(0.01, -0.02, "Source : NYC 311 Service Requests — janvier 2025", fontsize=8, color="gray")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lab3_figure_communication.png", dpi=150, bbox_inches="tight")



Adaptez évidemment le titre, l'annotation et les valeurs à **votre** graphique et à **vos** résultats : cette figure doit être unique à votre analyse, pas un simple copier-coller.

## 10. Travail demandé

### Partie A — Choisir avant de coder

1. Pour chacune des cinq familles de questions du chapitre 3 (distribution, comparaison, relation, évolution, composition), indiquez la variable ou la paire de variables de ce jeu de données que vous allez représenter, **avant** d'écrire le code.

### Partie B — Graphiques essentiels

2. Produisez l'histogramme et la boîte à moustaches du délai de traitement (section 3).
3. Produisez le diagramme en barres des 12 types de demandes les plus fréquents (section 4).
4. Produisez le camembert des 8 mêmes catégories et rédigez, en trois phrases maximum, pourquoi le diagramme en barres est ici préférable.
5. Produisez le nuage de points heure de création / délai, avec échantillonnage et transparence (section 5).
6. Produisez la heatmap de corrélation (section 5).
7. Produisez la courbe d'évolution quotidienne du volume de demandes (section 6).
8. Produisez la boîte à moustaches par arrondissement et la heatmap arrondissement/canal (section 7).

### Partie C — Critique et correction

9. Sur l'un de vos propres graphiques, tronquez volontairement l'axe des ordonnées d'un diagramme en barres et comparez les deux versions côte à côte. Expliquez en deux phrases l'effet produit.
10. Identifiez, parmi les huit graphiques produits, celui qui serait le plus à risque d'être mal interprété par un lecteur pressé, et proposez une correction (titre, annotation, ou axe).

### Partie D — Interactivité

11. Produisez le graphique interactif `Plotly` de la section 8.
12. Citez une situation, dans votre futur projet de fin de module, où un graphique interactif apporterait une réelle valeur par rapport à une image statique — et une situation où il n'en apporterait aucune.

### Partie E — Communication

13. Produisez la figure de communication complète de la section 9 (titre-message, annotation, source).
14. Rédigez, pour cette figure uniquement, une légende de 3 à 4 phrases destinée à un lecteur qui ne verra que le graphique et sa légende, sans lire le reste du rapport.
15. Constituez un court portfolio (un fichier Markdown ou un notebook) rassemblant vos huit graphiques, chacun accompagné d'un titre, d'une phrase de lecture et de la question à laquelle il répond.
